In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


In [3]:
df = pd.read_csv('/content/spam.csv', encoding='latin-1')

In [4]:
df.shape

(5572, 5)

In [5]:
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [6]:
df.isnull().sum()

,0
v1,0
v2,0
Unnamed: 2,5522
Unnamed: 3,5560
Unnamed: 4,5566


In [7]:
df.drop(columns=['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'])

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will Ì_ b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [9]:
df['v1'].value_counts()

,count
v1,
ham,4825
spam,747


In [10]:
df = df[['v1', 'v2']]

df.columns = ['label', 'text']

In [11]:
s = {'ham': 0, 'spam': 1}
df['label'] = df['label'].map(s)

In [12]:
df['text'] = df['text'].apply(lambda x: x.lower())

In [14]:
import string
def remove_punc(txt):
    return txt.translate(str.maketrans('', '', string.punctuation))
df['text'] = df['text'].apply(remove_punc)

In [15]:
def remove_numbers(txt):
    new = ""
    for i in txt:
        if not i.isdigit():
            new = new + i
    return new

df['text'] = df['text'].apply(remove_numbers)

In [16]:
def remove_emojis(txt):
    new = ""
    for i in txt:
        if i.isascii():
            new += i
    return new

df['text'] = df['text'].apply(remove_emojis)

In [17]:
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [18]:
def remove(txt):
    words = txt.split()
    cleaned = []

    for i in words:
        if i not in stop_words:
            cleaned.append(i)

    return ' '.join(cleaned)

df['text'] = df['text'].apply(remove)

In [19]:
from nltk.stem import WordNetLemmatizer
import nltk

nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()
def lemmatize_text(text):
    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words]
    return ' '.join(words)

[nltk_data] Downloading package wordnet to /root/nltk_data...


In [20]:
df['text'] = df['text'].apply(lemmatize_text)

In [21]:
df['text'].head()

,text
0,go jurong point crazy available bugis n great ...
1,ok lar joking wif u oni
2,free entry wkly comp win fa cup final tkts st ...
3,u dun say early hor u c already say
4,nah dont think go usf life around though


In [24]:
from sklearn.model_selection import train_test_split

In [25]:
X_train, X_test, y_train, y_test = train_test_split(
    df['text'],
    df['label'],
    test_size=0.2,
    random_state=42
)

In [27]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [28]:
tfidf = TfidfVectorizer()

X_train_tf = tfidf.fit_transform(X_train)
X_test_tf = tfidf.transform(X_test)

In [29]:
from sklearn.linear_model import LogisticRegression
lr=LogisticRegression()

In [30]:
lr = LogisticRegression(max_iter=1000)

lr.fit(X_train_tf, y_train)

LogisticRegression(max_iter=1000)

In [31]:
y_pred = lr.predict(X_test_tf)

In [32]:
from sklearn.metrics import accuracy_score, f1_score

In [33]:
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("F1 Score:", f1)

Accuracy: 0.9479820627802691
F1 Score: 0.768


In [34]:
words = np.array(tfidf.get_feature_names_out())
coefficients = lr.coef_[0]

In [35]:
top_spam = words[np.argsort(coefficients)[-10:]][::-1]

print("Top 10 Spam Words:")
print(top_spam)

Top 10 Spam Words:
['txt' 'free' 'claim' 'stop' 'mobile' 'call' 'service' 'reply' 'text'
 'prize']


In [36]:
top_ham = words[np.argsort(coefficients)[:10]]

print("Top 10 Ham Words:")
print(top_ham)

Top 10 Ham Words:
['ltgt' 'im' 'ill' 'ok' 'sir' 'later' 'da' 'going' 'home' 'come']
